In [15]:
import pandas as pd
import numpy as np
import seaborn as sns

# Option A: Built-in seaborn dataset (fastest in Colab)
df = sns.load_dataset('titanic')

# Option B: If you uploaded titanic.csv to Colab files
# df = pd.read_csv('titanic.csv')

print("Shape:", df.shape)
df.head()

Shape: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [16]:
df.info()
display(df.describe(include='all').T)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
survived,891.0,NaN,NaN,NaN,0.383838,0.486592,0.0,0.0,0.0,1.0,1.0
pclass,891.0,NaN,NaN,NaN,2.308642,0.836071,1.0,2.0,3.0,3.0,3.0
sex,891,2,male,577,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,714.0,NaN,NaN,NaN,29.699118,14.526497,0.42,20.125,28.0,38.0,80.0
sibsp,891.0,NaN,NaN,NaN,0.523008,1.102743,0.0,0.0,0.0,1.0,8.0
parch,891.0,NaN,NaN,NaN,0.381594,0.806057,0.0,0.0,0.0,0.0,6.0
fare,891.0,NaN,NaN,NaN,32.204208,49.693429,0.0,7.9104,14.4542,31.0,512.3292
embarked,889,3,S,644,NaN,NaN,NaN,NaN,NaN,NaN,NaN
class,891,3,Third,491,NaN,NaN,NaN,NaN,NaN,NaN,NaN
who,891,3,man,537,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# 1. Check missing values & duplicates
print("Missing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())

# 2. Drop exact duplicates
df_clean = df.drop_duplicates().copy()

# 3. Impute Age by class & sex median; Embarked by mode
df_clean['age'] = df_clean['age'].fillna(
    df_clean.groupby(['pclass', 'sex'])['age'].transform('median')
)
df_clean['embarked'] = df_clean['embarked'].fillna(df_clean['embarked'].mode()[0])

# 4. Handle sparse deck/cabin info and drop redundant columns if present
df_clean['has_deck'] = df_clean['deck'].notnull().astype(int)
df_clean = df_clean.drop(columns=['deck', 'embark_town', 'alive'], errors='ignore')

print("Remaining nulls:", df_clean.isnull().sum().sum())

Missing values:
 age            177
embarked         2
deck           688
embark_town      2
dtype: int64

Duplicate rows: 107
Remaining nulls: 0


In [18]:
# Family features
df_clean['family_size'] = df_clean['sibsp'] + df_clean['parch'] + 1
df_clean['is_alone'] = (df_clean['family_size'] == 1).astype(int)

# Aggregations
by_sex_class = df_clean.groupby(['pclass', 'sex'])['survived'].agg(
    count='count',
    survival_rate='mean'
).round(3)

by_family = df_clean.groupby('is_alone')['survived'].agg(
    count='count',
    survival_rate='mean'
).round(3)

display(by_sex_class)
display(by_family)



count  survival_rate
pclass sex                         
1      female     93          0.968
       male      121          0.372
2      female     73          0.918
       male       92          0.185
3      female    127          0.472
       male      278          0.158

,count,survival_rate
is_alone,,
0,338,0.512
1,446,0.336
